### Import das Classes

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.ProblemGeneratorP1 import ProblemaP1Generator
from src.ProblemaP3 import ProblemaP3

### Problema 3

No problema 3, reaproveitamos o sorteio de obstruções do problema 2, mas a condição de contorno muda.

Agora fixamos uma pressão de entrada `P_in` nos nós de fonte e verificamos se a rede consegue manter pressão suficiente nos demais nós.

O objetivo é encontrar a menor frequência de limpeza para que a probabilidade de falha seja menor que `x`.

A falha usada no código é:

$$
\min_{i \in \mathcal{C}} p_i < P_{min}
$$

onde $\mathcal{C}$ é o conjunto de nós monitorados, sem incluir os nós de entrada com pressão prescrita.


### Gerando a rede base do P1

In [ ]:
generator = ProblemaP1Generator(seed=42)  # fixa a aleatoriedade

prob = generator.generate(
    num_nodes=100,      # numero de nos da rede
    edge_prob=0.25,     # chance de criar aresta entre dois nos
    mu=1e-3,            # viscosidade da agua em pa s
    patm=0.0,           # pressao de referencia usada no p1
    q_mode="uniform",   # distribui as vazoes de forma uniforme
    single_sink=True    # usa apenas um no de saida
)

prob.setup()  # monta as matrizes do problema
prob.solve()  # resolve o p1 apenas para visualizar a rede base

print("P1 resolvido para visualizacao")
print("pressao minima no p1:", np.min(prob.p))
print("pressao maxima no p1:", np.max(prob.p))


In [ ]:
prob.plot(
    show_edge_labels=False,
    show_node_labels=True,
    layout='spring',
    node_value_attr='pressao',
    edge_value_attr='vazao'
)

### Tabela de manutenção do P3

In [ ]:
pd.DataFrame(ProblemaP3.DEFAULT_MAINTENANCE_TABLE)

### Parâmetros do P3

In [ ]:
alpha = 2.0    # resistencia dobra quando o cano esta obstruido
P_in = 1.0     # pressao fixa nos nos de entrada
P_min = 0.968  # pressao minima admissivel nos nos monitorados
x = 0.05       # probabilidade maxima de falha aceita, 5%
N = 1000       # numero de simulacoes de monte carlo
seed = 42      # fixa a aleatoriedade


### Criando o problema probabilístico inverso

In [ ]:
p3 = ProblemaP3(
    p1_instance=prob,  # problema p1 ja montado
    alpha=alpha,       # fator de obstrucao
    P_in=P_in,         # pressao fixa nos nos de entrada
    P_min=P_min,       # pressao minima admissivel
    x=x,               # risco maximo aceito
    n_samples=N,       # numero de simulacoes
    seed=seed          # fixa a aleatoriedade
)


### Rodando o P3

In [ ]:
best, results = p3.solve_inverse_problem(
    n_iter=N,           # simulacoes por frequencia
    confidence=0.95,    # nivel do intervalo de confianca
    seed=seed,          # fixa a aleatoriedade
    use_upper_ci=False  # usa a probabilidade estimada como criterio
)

### Resultados por frequência

In [ ]:
df = pd.DataFrame([
    {k: v for k, v in row.items() if k != "node_failure_probability"}
    for row in results
])

cols = [
    "frequencia",
    "r_prob",
    "custo",
    "P_in",
    "P_min",
    "P_fail",
    "IC_lower",
    "IC_upper",
    "mean_min_pressure",
    "std_min_pressure",
    "min_pressure_observed",
    "accepted",
]

df_resultados = df[[col for col in cols if col in df.columns]]
df_resultados


### Checagem de coerência

Com pressão de entrada fixa, o esperado é que a probabilidade de falha diminua quando a frequência aumenta, pois `r_prob` diminui.


In [ ]:
df_resultados[["frequencia", "r_prob", "P_fail", "mean_min_pressure", "accepted"]]


### Resposta do problema

In [ ]:
if best is None:
    print("Nenhuma frequencia da tabela satisfaz o criterio.")
else:
    print("frequencia minima:", best["frequencia"], "vezes por ano")
    print("r:", best["r_prob"])
    print("custo:", best["custo"], "kR$")
    print("probabilidade de falha:", best["P_fail"])
    print("intervalo de confianca:", (best["IC_lower"], best["IC_upper"]))

### Gráfico da probabilidade de falha

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(df["frequencia"], df["P_fail"], marker="o")
plt.axhline(x, linestyle="--", label="limite x")
plt.xlabel("frequencia de limpeza em vezes por ano")
plt.ylabel("probabilidade de falha")
plt.title("P3: probabilidade de falha por frequencia")
plt.grid(True)
plt.legend()
plt.show()

### Gráfico do custo

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(df["frequencia"], df["custo"], marker="o")
plt.xlabel("frequencia de limpeza em vezes por ano")
plt.ylabel("custo em kR$")
plt.title("P3: custo de manutencao")
plt.grid(True)
plt.show()

### Resumo

In [ ]:
p3.summary()

### Conclusão

O P3 usa a mesma rede do P1 e o mesmo sorteio de obstruções do P2, mas muda a condição de contorno: a pressão é fixada nos nós de entrada.

Para cada frequência de limpeza, o código usa o valor correspondente de `r_prob`, roda Monte Carlo e estima a probabilidade de algum nó monitorado ficar com pressão abaixo de `P_min`.

A solução é a primeira frequência da tabela que satisfaz:

$$
\widehat{P}_f < x
$$

Se a tabela estiver coerente, `P_fail` deve diminuir quando a frequência aumenta.
